# Notebook 05 — Phase 3: Pair Verification + EER Calibration + INT8 PTQ

**Phase:** 3 (model selection + quantization)  
**Purpose:** Take all three MobileFaceNet candidates (InsightFace baseline, 04a fine-tune, 04b scratch), evaluate them on a held-out Bollywood pair-verification test set, pick the EER winner, INT8-quantize it, verify the INT8 version doesn't degrade EER by more than 0.5%, and ship it as the final `mobilefacenet.tflite`. Also INT8-quantize ShuffleNet liveness.  
**Expected runtime:** ~15–25 minutes  
**GPU required:** No (CPU is fine — inference + quantization only)

## Pipeline

1. **Build a balanced pair-verification test set** from Bollywood val faces (90/10 split same as 04a/04b, same SEED → same val files). 5k positive + 5k negative pairs.
2. **Evaluate each FP32 candidate** via cosine-distance pair verification → ROC → AUC + EER + threshold at EER.
3. **Pick the winner** (lowest EER; tiebreak on AUC).
4. **Load the winner's Keras checkpoint** (saved in 04a/04b). For the InsightFace baseline, re-download and convert via `onnx2tf`.
5. **INT8 PTQ** with a 200-sample representative dataset of cropped Bollywood faces.
6. **Re-evaluate the INT8 model** against the same pair set. If EER degraded by > 0.5%, ship FP32 instead.
7. **ShuffleNet INT8 PTQ** (optional — needs the `.keras` checkpoint from Notebook 03). Falls back to FP32 if missing.
8. **Emit `threshold_calibration.json`** with the calibrated EER cosine-distance threshold for `shared_contracts/thresholds.json`.

## What you need before running

Attach to this Kaggle notebook:

1. **Bollywood Faces** dataset (`havingfun/100-bollywood-celebrity-faces`) — same as 04a/04b.
2. **A private Kaggle Dataset called `mobilefacenet-candidates`** containing the artifacts from both 04a and 04b runs. Suggested structure:
   ```
   mobilefacenet-candidates/
   ├── 04a/
   │   ├── mobilefacenet_bollywood_ft.tflite
   │   └── best_finetune.keras
   ├── 04b/
   │   ├── mobilefacenet_bollywood_scratch.tflite
   │   └── best_scratch.keras
   └── 03/
       └── shufflenet_liveness.keras   (optional — for ShuffleNet INT8 PTQ)
   ```
   The notebook also handles missing candidates gracefully — if 04a's .tflite/keras isn't there, it's skipped and we evaluate just baseline + 04b.

## What to send back to Claude

Paste the literal output of: cell 7 (discovery), cell 9 (cropping), cell 11 (pair list), cell 15 (per-candidate eval), cell 17 (comparison + winner), cell 21 (INT8 PTQ size), cell 23 (INT8 EER vs FP32), cell 25 (ShuffleNet PTQ), cell 27 (final summary). If anything fails, paste full traceback.

## 1 — Config

Adjust the candidate paths after you upload `mobilefacenet-candidates` as a Kaggle Dataset. The notebook tries a few common mount patterns first; only edit if Kaggle gives you a different path.

In [ ]:
import os

# Reproducibility — MUST match 04a/04b SEED so val split is identical
SEED                 = 42
VAL_SPLIT            = 0.1
IMAGE_SIZE           = (112, 112)

# Pair verification
N_POS_PAIRS_TARGET   = 5000
N_NEG_PAIRS_TARGET   = 5000
EER_DEGRADATION_CAP  = 0.005   # ship INT8 only if EER worsens by < 0.5 pp

# INT8 PTQ
N_REPR_SAMPLES       = 200

# Workspaces
WORK_DIR    = "/kaggle/working"
MODELS_OUT  = f"{WORK_DIR}/models"
REPORTS_OUT = f"{WORK_DIR}/reports"
CROPS_DIR   = f"{WORK_DIR}/crops"
for d in (MODELS_OUT, REPORTS_OUT, CROPS_DIR):
    os.makedirs(d, exist_ok=True)

print("Config locked.")

## 2 — Clone repo + install

We need `sklearn` for ROC/AUC, plus `onnx2tf` only if the InsightFace baseline ends up winning (so we can re-convert for INT8 PTQ).

In [ ]:
import subprocess

REPO_URL = "https://github.com/MalayM09/nhai.git"
REPO_DIR = os.path.join(WORK_DIR, "nhai")
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print("CWD:", os.getcwd())
print(subprocess.run(["git", "log", "--oneline", "-5"], capture_output=True, text=True).stdout)

print("\nInstalling deps…")
!pip install -q scikit-learn

## 3 — Discover datasets + candidates

Locate the Bollywood dataset (same as 04a/04b) and the candidate `.tflite` + `.keras` files. The baseline `mobilefacenet.tflite` lives in the repo we just cloned.

In [ ]:
import glob

print("/kaggle/input contents:")
for d in sorted(os.listdir("/kaggle/input")):
    print(f"  {d}")

# Bollywood dataset (same discovery logic as 04a/04b/04c)
BOLLYWOOD_ROOT = None
for guess in [
    "/kaggle/input/100-bollywood-celebrity-faces",
    "/kaggle/input/datasets/havingfun/100-bollywood-celebrity-faces",
]:
    if os.path.isdir(guess):
        BOLLYWOOD_ROOT = guess; break
if BOLLYWOOD_ROOT is None:
    hits = glob.glob("/kaggle/input/**/bollywood_celeb_faces*", recursive=True)
    if hits:
        BOLLYWOOD_ROOT = os.path.dirname(hits[0])
assert BOLLYWOOD_ROOT, "Could not locate Bollywood dataset"
print(f"\nBollywood root: {BOLLYWOOD_ROOT}")

# Candidate models — Kaggle Dataset has all files flat at the dataset root
# (not nested in 04a/04b/04c/ subfolders). Discover by exact filename.
CANDIDATES_ROOT = None
for guess in [
    "/kaggle/input/mobilefacenet-candidates",
    "/kaggle/input/datasets/malaymishra6969/mobilefacenet-candidates",
    "/kaggle/input/datasets/malaymishra/mobilefacenet-candidates",
]:
    if os.path.isdir(guess):
        CANDIDATES_ROOT = guess; break
if CANDIDATES_ROOT is None:
    # Glob fallback: find any of the expected filenames anywhere under /kaggle/input
    for fname in ["mobilefacenet_bollywood_scratch.tflite",
                  "mobilefacenet_adapter.tflite",
                  "mobilefacenet_bollywood_ft.tflite"]:
        hits = glob.glob(f"/kaggle/input/**/{fname}", recursive=True)
        if hits:
            CANDIDATES_ROOT = os.path.dirname(hits[0])
            break

BASELINE_TFLITE = os.path.join(REPO_DIR, "mobile_app/assets/models/mobilefacenet.tflite")
assert os.path.isfile(BASELINE_TFLITE), f"baseline missing: {BASELINE_TFLITE}"

def find_in_candidates(filename):
    """Return path if file exists in CANDIDATES_ROOT (flat or any subfolder), else None."""
    if not CANDIDATES_ROOT:
        return None
    direct = os.path.join(CANDIDATES_ROOT, filename)
    if os.path.isfile(direct):
        return direct
    # Recursive fallback in case the user did nest folders
    hits = glob.glob(f"{CANDIDATES_ROOT}/**/{filename}", recursive=True)
    return hits[0] if hits else None

candidates = {
    "baseline_insightface": {
        "type":   "single",
        "tflite": BASELINE_TFLITE,
        "keras":  None,
        "note":   "InsightFace w600k_mbf pretrained, FP32 13 MB",
    },
}

if CANDIDATES_ROOT:
    print(f"\nCandidates root: {CANDIDATES_ROOT}")
    print(f"Files in candidates root:")
    for f in sorted(os.listdir(CANDIDATES_ROOT)):
        size_mb = os.path.getsize(os.path.join(CANDIDATES_ROOT, f)) / 1024 / 1024
        print(f"  {f}  ({size_mb:.2f} MB)")

    # 04a fine-tune (single tflite)
    a_tfl = find_in_candidates("mobilefacenet_bollywood_ft.tflite")
    a_ker = find_in_candidates("best_finetune.keras")
    if a_tfl:
        candidates["finetune_04a"] = {
            "type":   "single",
            "tflite": a_tfl,
            "keras":  a_ker,
            "note":   "04a fine-tune on Bollywood",
        }

    # 04b from-scratch (single tflite)
    b_tfl = find_in_candidates("mobilefacenet_bollywood_scratch.tflite")
    b_ker = find_in_candidates("best_scratch.keras")
    if b_tfl:
        candidates["scratch_04b"] = {
            "type":   "single",
            "tflite": b_tfl,
            "keras":  b_ker,
            "note":   "04b from-scratch on Bollywood",
        }

    # 04c adapter (COMPOSED: backbone + adapter tflite)
    c_tfl = find_in_candidates("mobilefacenet_adapter.tflite")
    c_ker = find_in_candidates("best_adapter.keras")
    if c_tfl:
        candidates["adapter_04c"] = {
            "type":             "composed",
            "tflite":           c_tfl,
            "backbone_tflite":  BASELINE_TFLITE,
            "keras":            c_ker,
            "note":             "04c frozen baseline + adapter (composed pipeline)",
        }
else:
    print("\nNo candidates dataset found — only baseline will be evaluated.")

print("\nFinal candidate list:")
for name, c in candidates.items():
    kstatus = "yes" if c["keras"] else "no"
    if c["type"] == "composed":
        print(f"  {name:25s} COMPOSED  backbone={os.path.basename(c['backbone_tflite'])} + adapter={os.path.basename(c['tflite'])}  keras={kstatus}")
    else:
        print(f"  {name:25s} single    {os.path.basename(c['tflite'])}  keras={kstatus}")

# ShuffleNet keras checkpoint (optional)
SHUFFLENET_KERAS = find_in_candidates("shufflenet_liveness.keras")
if SHUFFLENET_KERAS:
    print(f"\nShuffleNet keras checkpoint: {SHUFFLENET_KERAS} (INT8 PTQ will run)")
else:
    print("\nShuffleNet keras NOT found — ShuffleNet INT8 PTQ will be skipped, FP32 ships.")

## 4 — Build the held-out test face crops

Walk the same 3 Bollywood split folders as 04a/04b. Apply the **same SEED** for the per-celeb train/val split → the val files here are exactly the val files there. Crop with OpenCV Haar cascade (same as 04a/04b).

In [ ]:
import cv2, numpy as np, random
from collections import defaultdict
from tqdm.auto import tqdm

# Discover the 3 split folders
SPLIT_FOLDERS = sorted([os.path.join(BOLLYWOOD_ROOT, d) for d in os.listdir(BOLLYWOOD_ROOT)
                       if os.path.isdir(os.path.join(BOLLYWOOD_ROOT, d)) and d.startswith("bollywood_celeb_faces")])

# Gather images per celebrity
IMG_EXTS = (".jpg", ".jpeg", ".png")
celeb_to_imgs = defaultdict(list)
for split in SPLIT_FOLDERS:
    for celeb in sorted(os.listdir(split)):
        cdir = os.path.join(split, celeb)
        if not os.path.isdir(cdir): continue
        for f in os.listdir(cdir):
            if f.lower().endswith(IMG_EXTS):
                celeb_to_imgs[celeb].append(os.path.join(cdir, f))
celebs = sorted(celeb_to_imgs.keys())
print(f"Total celebs: {len(celebs)} | total images: {sum(len(v) for v in celeb_to_imgs.values())}")

# Per-celeb train/val split with SAME SEED -> reproduces 04a/04b val files exactly
rng = random.Random(SEED)
val_files_per_celeb = {}
for celeb in celebs:
    imgs = celeb_to_imgs[celeb][:]
    rng.shuffle(imgs)
    n_val = max(1, int(len(imgs) * VAL_SPLIT))
    val_files_per_celeb[celeb] = imgs[:n_val]
val_total = sum(len(v) for v in val_files_per_celeb.values())
print(f"Val (held-out) images: {val_total}")

# Crop val faces (Haar)
CASCADE = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
assert not CASCADE.empty()

def detect_and_crop(img_path, out_size=112, margin=0.2):
    img_bgr = cv2.imread(img_path)
    if img_bgr is None: return None
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    h, w = img_rgb.shape[:2]
    faces = CASCADE.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=4, minSize=(20, 20))
    if len(faces) > 0:
        x, y, fw, fh = max(faces, key=lambda f: f[2] * f[3])
        mx = int(fw * margin); my = int(fh * margin)
        x0 = max(0, x - mx); y0 = max(0, y - my)
        x1 = min(w, x + fw + mx); y1 = min(h, y + fh + my)
        face = img_rgb[y0:y1, x0:x1]
    else:
        s = min(h, w)
        face = img_rgb[(h-s)//2:(h+s)//2, (w-s)//2:(w+s)//2]
    return cv2.resize(face, (out_size, out_size), interpolation=cv2.INTER_LINEAR)

crop_paths_per_celeb = defaultdict(list)
all_pairs = [(c, p) for c in celebs for p in val_files_per_celeb[c]]
for celeb, src in tqdm(all_pairs, desc="cropping val faces"):
    out_dir = os.path.join(CROPS_DIR, celeb); os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, os.path.splitext(os.path.basename(src))[0] + ".jpg")
    if not os.path.isfile(out_path):
        face = detect_and_crop(src)
        if face is None: continue
        cv2.imwrite(out_path, cv2.cvtColor(face, cv2.COLOR_RGB2BGR))
    crop_paths_per_celeb[celeb].append(out_path)

n_total = sum(len(v) for v in crop_paths_per_celeb.values())
print(f"\nCropped val faces: {n_total} across {len(crop_paths_per_celeb)} celebs")
print(f"Min/median/max per celeb: {min(len(v) for v in crop_paths_per_celeb.values())}/"
      f"{sorted(len(v) for v in crop_paths_per_celeb.values())[len(crop_paths_per_celeb)//2]}/"
      f"{max(len(v) for v in crop_paths_per_celeb.values())}")

## 5 — Build balanced pair lists

- **Positive pairs:** all C(N, 2) pairs per celeb, sampled to hit `N_POS_PAIRS_TARGET`.
- **Negative pairs:** random (celeb_A, img_A) × (celeb_B, img_B) where A ≠ B.

Balanced 5k/5k by default.

In [ ]:
from itertools import combinations

rng = random.Random(SEED + 999)

# Positive pairs
pos_candidates = []
for celeb, paths in crop_paths_per_celeb.items():
    if len(paths) >= 2:
        pos_candidates.extend(combinations(paths, 2))
rng.shuffle(pos_candidates)
pos_pairs = pos_candidates[:N_POS_PAIRS_TARGET]

# Negative pairs
celebs_with_imgs = [c for c, p in crop_paths_per_celeb.items() if len(p) > 0]
neg_pairs = []
while len(neg_pairs) < N_NEG_PAIRS_TARGET:
    c1, c2 = rng.sample(celebs_with_imgs, 2)
    p1 = rng.choice(crop_paths_per_celeb[c1])
    p2 = rng.choice(crop_paths_per_celeb[c2])
    neg_pairs.append((p1, p2))

print(f"Positive pairs: {len(pos_pairs):,}")
print(f"Negative pairs: {len(neg_pairs):,}")
print(f"Total pairs:    {len(pos_pairs) + len(neg_pairs):,}")

# How many unique images will each model embed?
all_paths_in_pairs = set()
for p1, p2 in pos_pairs + neg_pairs:
    all_paths_in_pairs.add(p1); all_paths_in_pairs.add(p2)
print(f"Unique images to embed per model: {len(all_paths_in_pairs):,}")

## 6 — Embedding extractor + ROC/EER framework

Reusable helpers. For each `.tflite` candidate: embed every unique image once, then compute pairwise cosine distance. `roc_auc_score` over `-distances` (higher score → more likely same person), then find EER via FPR/FNR crossover.

In [ ]:
import tensorflow as tf
from sklearn.metrics import roc_curve, roc_auc_score

def load_interpreter(tflite_path):
    it = tf.lite.Interpreter(model_path=tflite_path)
    it.allocate_tensors()
    inp = it.get_input_details()[0]
    out = it.get_output_details()[0]
    return it, inp, out

def preprocess_for_mfn(img_path):
    """Read cropped face -> [-1, 1] RGB as MobileFaceNet expects."""
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, IMAGE_SIZE).astype(np.float32)
    img = (img - 127.5) / 127.5
    return img

def embed_all_single(tflite_path, paths, normalize=True):
    """Single-tflite embedding: image -> tflite -> 512-D."""
    it, inp, out = load_interpreter(tflite_path)
    embs = {}
    for p in tqdm(paths, desc=f"embed {os.path.basename(tflite_path)}", leave=False):
        x = preprocess_for_mfn(p)[None].astype(inp["dtype"])
        it.set_tensor(inp["index"], x)
        it.invoke()
        e = it.get_tensor(out["index"])[0]
        if normalize:
            e = e / (np.linalg.norm(e) + 1e-9)
        embs[p] = e
    return embs

def embed_all_composed(backbone_tflite, adapter_tflite, paths, normalize=True):
    """Composed embedding: image -> backbone -> 512-D -> adapter -> 512-D adapted."""
    it_b, inp_b, out_b = load_interpreter(backbone_tflite)
    it_a, inp_a, out_a = load_interpreter(adapter_tflite)
    embs = {}
    for p in tqdm(paths, desc=f"embed composed (backbone+{os.path.basename(adapter_tflite)})", leave=False):
        x = preprocess_for_mfn(p)[None].astype(inp_b["dtype"])
        # Step 1: image -> backbone -> 512-D raw embedding
        it_b.set_tensor(inp_b["index"], x)
        it_b.invoke()
        e_base = it_b.get_tensor(out_b["index"])
        # Step 2: 512-D raw -> adapter -> 512-D adapted
        it_a.set_tensor(inp_a["index"], e_base.astype(inp_a["dtype"]))
        it_a.invoke()
        e_adapted = it_a.get_tensor(out_a["index"])[0]
        if normalize:
            e_adapted = e_adapted / (np.linalg.norm(e_adapted) + 1e-9)
        embs[p] = e_adapted
    return embs

def cosine_dist(a, b):
    return 1.0 - float(np.dot(a, b))

def evaluate_model(candidate_meta, pos_pairs, neg_pairs):
    """Dispatches on candidate type. Returns dict with auc, eer, threshold, etc."""
    unique = set()
    for p1, p2 in pos_pairs + neg_pairs:
        unique.add(p1); unique.add(p2)
    unique = sorted(unique)
    
    if candidate_meta["type"] == "single":
        embs = embed_all_single(candidate_meta["tflite"], unique)
        size_mb = os.path.getsize(candidate_meta["tflite"]) / 1024 / 1024
    elif candidate_meta["type"] == "composed":
        embs = embed_all_composed(candidate_meta["backbone_tflite"],
                                   candidate_meta["tflite"], unique)
        size_mb = (os.path.getsize(candidate_meta["backbone_tflite"]) +
                   os.path.getsize(candidate_meta["tflite"])) / 1024 / 1024
    else:
        raise ValueError(f"unknown candidate type: {candidate_meta['type']}")
    
    pos_d = np.array([cosine_dist(embs[a], embs[b]) for a, b in pos_pairs])
    neg_d = np.array([cosine_dist(embs[a], embs[b]) for a, b in neg_pairs])
    
    labels = np.concatenate([np.ones(len(pos_d)), np.zeros(len(neg_d))])
    scores = -np.concatenate([pos_d, neg_d])  # higher = more likely same
    
    auc = float(roc_auc_score(labels, scores))
    fpr, tpr, thr = roc_curve(labels, scores)
    fnr = 1 - tpr
    eer_idx = int(np.argmin(np.abs(fpr - fnr)))
    eer = float((fpr[eer_idx] + fnr[eer_idx]) / 2)
    threshold_at_eer = float(-thr[eer_idx])
    
    return {
        "auc": auc,
        "eer": eer,
        "threshold_at_eer": threshold_at_eer,
        "pos_mean_dist": float(pos_d.mean()),
        "neg_mean_dist": float(neg_d.mean()),
        "size_mb": size_mb,
        "type": candidate_meta["type"],
        "_fpr": fpr, "_tpr": tpr,
    }

print("Helpers defined (single + composed evaluators).")

In [ ]:
results = {}
for name, meta in candidates.items():
    print(f"\n=== Evaluating: {name}  ({meta['note']}) ===")
    results[name] = evaluate_model(meta, pos_pairs, neg_pairs)
    r = results[name]
    print(f"  Type: {r['type']}")
    print(f"  Size: {r['size_mb']:.2f} MB")
    print(f"  AUC:  {r['auc']:.4f}")
    print(f"  EER:  {r['eer']:.4f}  (threshold @ EER = {r['threshold_at_eer']:.4f} cosine distance)")
    print(f"  Mean pos-pair distance: {r['pos_mean_dist']:.4f}")
    print(f"  Mean neg-pair distance: {r['neg_mean_dist']:.4f}")

## 8 — Comparison + winner

Pick the EER winner (tiebreak on AUC). Print a clean comparison.

In [ ]:
print("=" * 78)
print(f"{'Model':<25}{'Size (MB)':<12}{'AUC':<10}{'EER':<12}{'Thr @EER'}")
print("=" * 78)
for name, r in sorted(results.items(), key=lambda kv: kv[1]["eer"]):
    print(f"{name:<25}{r['size_mb']:<12.2f}{r['auc']:<10.4f}{r['eer']:<12.4f}{r['threshold_at_eer']:.4f}")

# Pick winner
WINNER_NAME = min(results.keys(), key=lambda n: (results[n]["eer"], -results[n]["auc"]))
WINNER = results[WINNER_NAME]
print(f"\n>>> WINNER: {WINNER_NAME}  (EER={WINNER['eer']:.4f}, AUC={WINNER['auc']:.4f}) <<<")

WINNER_TFLITE = candidates[WINNER_NAME]["tflite"]
WINNER_KERAS  = candidates[WINNER_NAME]["keras"]
print(f"\nWinner .tflite: {WINNER_TFLITE}")
print(f"Winner .keras:  {WINNER_KERAS or '(none — INT8 PTQ will fall back to handling)'}")

## 9 — INT8 PTQ on the winning MobileFaceNet

Three paths depending on what the winner is:

1. **Winner is 04a or 04b:** load the saved `.keras` checkpoint → PTQ directly.
2. **Winner is the baseline:** re-download InsightFace, re-convert ONNX → SavedModel via `onnx2tf`, load as Keras, PTQ.
3. **No `.keras` available + winner is baseline + onnx2tf fails:** ship the FP32 baseline as-is (still under the 20 MB models cap).

Representative dataset: 200 random cropped Bollywood val faces, same normalization as inference (`[-1, 1]`).

In [ ]:
import keras

# Representative dataset for PTQ — sample N_REPR_SAMPLES cropped val faces
all_crops = [p for v in crop_paths_per_celeb.values() for p in v]
rng2 = random.Random(SEED + 7)
rng2.shuffle(all_crops)
repr_paths = all_crops[:N_REPR_SAMPLES]
print(f"Representative dataset: {len(repr_paths)} cropped Bollywood val faces")

def repr_dataset_gen():
    for p in repr_paths:
        x = preprocess_for_mfn(p)[None].astype(np.float32)
        yield [x]

# Get the Keras model to quantize
keras_model_for_ptq = None
ptq_source = None

if WINNER_KERAS and os.path.isfile(WINNER_KERAS):
    print(f"\nLoading Keras checkpoint: {WINNER_KERAS}")
    # The .keras file is a training_model with (image, label) inputs.
    # We need the backbone — fortunately it's the first input/Model.
    full = keras.models.load_model(WINNER_KERAS, compile=False, safe_mode=False)
    # The backbone is the layer that takes the image input and produces the embedding.
    # Find it by looking for the Functional model with image -> 512-D output.
    backbone = None
    for layer in full.layers:
        if isinstance(layer, keras.Model) and layer.output_shape[-1] == 512:
            backbone = layer; break
    if backbone is None:
        # Fallback: rebuild backbone-only by tracing the image input
        img_in = keras.layers.Input(shape=(112, 112, 3))
        # Find the backbone by walking layers connected to the image input
        # This is fragile; if it fails, ship FP32
        print("Could not auto-extract backbone — shipping FP32")
    else:
        inf_in = keras.layers.Input(shape=(112, 112, 3))
        inf_out = backbone(inf_in)
        keras_model_for_ptq = keras.Model(inf_in, inf_out)
        ptq_source = f"keras checkpoint ({WINNER_NAME})"
        print(f"Backbone extracted. Params: {keras_model_for_ptq.count_params():,}")

elif WINNER_NAME == "baseline_insightface":
    # Re-download buffalo_s, convert to SavedModel, load as Keras
    print("\nWinner is the InsightFace baseline. Re-downloading + converting via onnx2tf…")
    !pip install -q onnx2tf onnx onnxruntime onnx_graphsurgeon sng4onnx
    import urllib.request, zipfile
    bz = f"{WORK_DIR}/buffalo_s.zip"
    bd = f"{WORK_DIR}/buffalo_s"
    if not os.path.isfile(bz):
        urllib.request.urlretrieve("https://github.com/deepinsight/insightface/releases/download/v0.7/buffalo_s.zip", bz)
    if not os.path.isdir(bd):
        os.makedirs(bd, exist_ok=True)
        with zipfile.ZipFile(bz) as z: z.extractall(bd)
    onnx_path = None
    for root, _, files in os.walk(bd):
        for f in files:
            if f == "w600k_mbf.onnx":
                onnx_path = os.path.join(root, f)
    out_dir = f"{WORK_DIR}/baseline_keras"
    import shutil; shutil.rmtree(out_dir, ignore_errors=True)
    !onnx2tf -i {onnx_path} -o {out_dir} -b 1 -ois input.1:1,3,112,112
    
    sm_candidates = [out_dir] + [os.path.join(out_dir, d) for d in os.listdir(out_dir)
                                 if os.path.isdir(os.path.join(out_dir, d))]
    for sm in sm_candidates:
        if not os.path.isfile(os.path.join(sm, "saved_model.pb")):
            continue
        try:
            keras_model_for_ptq = tf.keras.models.load_model(sm)
            ptq_source = f"baseline re-converted from ONNX ({sm})"
            break
        except Exception as e:
            print(f"  load_model failed on {sm}: {e}")

print(f"\nPTQ source: {ptq_source}")

In [ ]:
WINNER_INT8 = f"{MODELS_OUT}/mobilefacenet.tflite"
WINNER_FP32_COPY = f"{MODELS_OUT}/mobilefacenet_fp32_backup.tflite"
import shutil
shutil.copy(WINNER_TFLITE, WINNER_FP32_COPY)
print(f"Backed up FP32 winner -> {WINNER_FP32_COPY}")

if keras_model_for_ptq is not None:
    print("\nRunning INT8 PTQ…")
    converter = tf.lite.TFLiteConverter.from_keras_model(keras_model_for_ptq)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = repr_dataset_gen
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.float32   # keep input float, convert internally
    converter.inference_output_type = tf.float32
    int8_bytes = converter.convert()
    with open(WINNER_INT8, "wb") as f:
        f.write(int8_bytes)
    print(f"INT8 TFLite saved: {WINNER_INT8}  ({os.path.getsize(WINNER_INT8)/1024/1024:.2f} MB)")
    print(f"vs FP32:           {os.path.getsize(WINNER_TFLITE)/1024/1024:.2f} MB")
else:
    print("\nNo PTQ source available. Shipping FP32 winner directly.")
    shutil.copy(WINNER_TFLITE, WINNER_INT8)
    print(f"FP32 copied as final: {WINNER_INT8}  ({os.path.getsize(WINNER_INT8)/1024/1024:.2f} MB)")

## 10 — Verify INT8 EER vs FP32

Re-run pair verification on the INT8 model. If EER worsens by more than `EER_DEGRADATION_CAP` (0.5%), restore the FP32 backup as the final ship.

In [ ]:
if WINNER_INT8 != WINNER_TFLITE and os.path.getsize(WINNER_INT8) < os.path.getsize(WINNER_TFLITE):
    # We actually quantized — verify
    int8_results = evaluate_model(WINNER_INT8, pos_pairs, neg_pairs)
    print(f"\nINT8 results vs FP32:")
    print(f"  Size:  {int8_results['size_mb']:.2f} MB  (was {WINNER['size_mb']:.2f} MB FP32)")
    print(f"  AUC:   {int8_results['auc']:.4f}  (was {WINNER['auc']:.4f})")
    print(f"  EER:   {int8_results['eer']:.4f}  (was {WINNER['eer']:.4f})")
    print(f"  Δ EER: {int8_results['eer'] - WINNER['eer']:+.4f}")
    
    eer_degradation = int8_results["eer"] - WINNER["eer"]
    if eer_degradation > EER_DEGRADATION_CAP:
        print(f"\nINT8 degraded EER by {eer_degradation:.4f} > {EER_DEGRADATION_CAP} cap. Falling back to FP32.")
        shutil.copy(WINNER_FP32_COPY, WINNER_INT8)
        FINAL_RESULTS = WINNER
        FINAL_PRECISION = "FP32 (INT8 degraded > cap)"
    else:
        print(f"\nINT8 within tolerance. Shipping INT8.")
        FINAL_RESULTS = int8_results
        FINAL_PRECISION = "INT8"
else:
    FINAL_RESULTS = WINNER
    FINAL_PRECISION = "FP32 (no PTQ applied)"

print(f"\nFinal MobileFaceNet precision: {FINAL_PRECISION}")
print(f"Final size: {os.path.getsize(WINNER_INT8)/1024/1024:.2f} MB")

## 11 — ShuffleNet INT8 PTQ (optional)

If the user attached the ShuffleNet `.keras` checkpoint from Notebook 03, INT8-quantize it. Representative dataset: 200 cropped Bollywood faces (they're faces, similar distribution to CelebA-Spoof live samples — not ideal but close enough for PTQ calibration).

In [ ]:
SHUFFLENET_FP32_PATH = os.path.join(REPO_DIR, "mobile_app/assets/models/shufflenet_liveness.tflite")
SHUFFLENET_OUT = f"{MODELS_OUT}/shufflenet_liveness.tflite"

if SHUFFLENET_KERAS and os.path.isfile(SHUFFLENET_KERAS):
    print(f"Quantizing ShuffleNet from {SHUFFLENET_KERAS}")
    # ShuffleNet expects [0, 1] range, not [-1, 1]
    def repr_dataset_gen_shufflenet():
        for p in repr_paths:
            img = cv2.imread(p)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, IMAGE_SIZE).astype(np.float32) / 255.0   # [0, 1]
            yield [img[None]]
    sh_model = keras.models.load_model(SHUFFLENET_KERAS, compile=False, safe_mode=False,
                                      custom_objects={"ChannelShuffle": None})
    # If ChannelShuffle custom_objects=None breaks loading, ship FP32
    try:
        converter = tf.lite.TFLiteConverter.from_keras_model(sh_model)
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = repr_dataset_gen_shufflenet
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type = tf.float32
        converter.inference_output_type = tf.float32
        sh_bytes = converter.convert()
        with open(SHUFFLENET_OUT, "wb") as f:
            f.write(sh_bytes)
        sh_size = os.path.getsize(SHUFFLENET_OUT) / 1024 / 1024
        print(f"ShuffleNet INT8: {sh_size:.2f} MB  (was {os.path.getsize(SHUFFLENET_FP32_PATH)/1024/1024:.2f} MB FP32)")
    except Exception as e:
        print(f"ShuffleNet PTQ failed: {e}")
        print("Shipping ShuffleNet FP32.")
        shutil.copy(SHUFFLENET_FP32_PATH, SHUFFLENET_OUT)
else:
    print("ShuffleNet .keras not attached. Shipping FP32 ShuffleNet as-is.")
    shutil.copy(SHUFFLENET_FP32_PATH, SHUFFLENET_OUT)

print(f"Final ShuffleNet: {SHUFFLENET_OUT}  ({os.path.getsize(SHUFFLENET_OUT)/1024/1024:.2f} MB)")

## 12 — Emit `threshold_calibration.json` and final summary

This JSON is what gets merged into `shared_contracts/thresholds.json` on the laptop. It contains the calibrated cosine-distance EER threshold and metadata.

In [ ]:
import json

THRESHOLD_OUT = f"{REPORTS_OUT}/threshold_calibration.json"
threshold_data = {
    "match_threshold_metric":          "cosine_distance",
    "match_threshold_value":           round(FINAL_RESULTS["threshold_at_eer"], 4),
    "match_threshold_status":          "calibrated",
    "match_threshold_calibrated_from": WINNER_NAME,
    "calibration_eer":                 round(FINAL_RESULTS["eer"], 4),
    "calibration_auc":                 round(FINAL_RESULTS["auc"], 4),
    "calibration_pairs_pos":           len(pos_pairs),
    "calibration_pairs_neg":           len(neg_pairs),
    "calibration_dataset":             "bollywood-100-celebrities-val-split (SEED=42, VAL_SPLIT=0.1)",
    "final_precision":                 FINAL_PRECISION,
    "final_size_mb":                   round(os.path.getsize(WINNER_INT8) / 1024 / 1024, 2),
    "all_candidates":                  {n: {"auc": r["auc"], "eer": r["eer"], "size_mb": r["size_mb"]}
                                        for n, r in results.items()},
}
with open(THRESHOLD_OUT, "w") as f:
    json.dump(threshold_data, f, indent=2)
print(f"Threshold calibration JSON saved: {THRESHOLD_OUT}")
print(json.dumps(threshold_data, indent=2))

## 13 — Plots: ROC overlay + distance distributions

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC overlay
for name, r in results.items():
    axes[0].plot(r["_fpr"], r["_tpr"], label=f"{name} (AUC={r['auc']:.3f}, EER={r['eer']:.3f})")
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.3)
axes[0].set_xlabel("False Positive Rate"); axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC curves (Bollywood val pair verification)")
axes[0].legend(loc="lower right"); axes[0].grid(alpha=0.3)

# Distance distributions for the winner
winner_embs = embed_all(WINNER_INT8, sorted({p for pair in pos_pairs + neg_pairs for p in pair}))
winner_pos = np.array([cosine_dist(winner_embs[a], winner_embs[b]) for a, b in pos_pairs])
winner_neg = np.array([cosine_dist(winner_embs[a], winner_embs[b]) for a, b in neg_pairs])
axes[1].hist(winner_pos, bins=40, alpha=0.6, label="same person", color="green")
axes[1].hist(winner_neg, bins=40, alpha=0.6, label="different person", color="red")
axes[1].axvline(FINAL_RESULTS["threshold_at_eer"], color="black", linestyle="--", label=f"EER threshold = {FINAL_RESULTS['threshold_at_eer']:.3f}")
axes[1].set_xlabel("Cosine distance"); axes[1].set_ylabel("Count")
axes[1].set_title(f"Final shipped model: {WINNER_NAME} ({FINAL_PRECISION})")
axes[1].legend(); axes[1].grid(alpha=0.3)

fig.tight_layout()
ROC_PLOT = f"{REPORTS_OUT}/pair_verification_roc_curves.png"
fig.savefig(ROC_PLOT, dpi=120, bbox_inches="tight")
plt.show()
print(f"Plot saved: {ROC_PLOT}")

# Final summary table
print("\n" + "=" * 78)
print("NOTEBOOK 05 — FINAL SUMMARY")
print("=" * 78)
print(f"Pair verification test set: {len(pos_pairs):,} pos + {len(neg_pairs):,} neg")
print()
for n, r in sorted(results.items(), key=lambda kv: kv[1]["eer"]):
    marker = " <-- winner" if n == WINNER_NAME else ""
    print(f"  {n:<25} AUC={r['auc']:.4f}  EER={r['eer']:.4f}  size={r['size_mb']:.2f} MB{marker}")
print()
print(f"Shipped MobileFaceNet:    {WINNER_INT8}  ({FINAL_PRECISION}, {os.path.getsize(WINNER_INT8)/1024/1024:.2f} MB)")
print(f"Shipped ShuffleNet:       {SHUFFLENET_OUT}  ({os.path.getsize(SHUFFLENET_OUT)/1024/1024:.2f} MB)")
print(f"Calibrated EER threshold: {FINAL_RESULTS['threshold_at_eer']:.4f} cosine distance")
print(f"Threshold JSON:           {THRESHOLD_OUT}")
print(f"ROC plot:                 {ROC_PLOT}")

## What to send back to Claude

Paste the literal output of cells 7, 9, 11, 15, 17, 21, 23, 25, 27 (discovery, cropping, pairs, per-candidate eval, comparison, INT8 PTQ size, INT8 EER vs FP32, ShuffleNet PTQ, final summary). If anything fails, paste the full traceback.

## What to do after the run

Download to laptop into `kaggle_downloads/05_eer_calibration/`:

- `/kaggle/working/models/mobilefacenet.tflite`        (final shipped model — INT8 or FP32 depending on degradation check)
- `/kaggle/working/models/shufflenet_liveness.tflite`  (INT8 if attached, else FP32 unchanged)
- `/kaggle/working/reports/threshold_calibration.json` (for `shared_contracts/thresholds.json` update)
- `/kaggle/working/reports/pair_verification_roc_curves.png`

Then on the laptop:

```bash
# Replace the placeholder MobileFaceNet with the calibrated/quantized winner
mv kaggle_downloads/05_eer_calibration/mobilefacenet.tflite       mobile_app/assets/models/
mv kaggle_downloads/05_eer_calibration/shufflenet_liveness.tflite mobile_app/assets/models/
mv kaggle_downloads/05_eer_calibration/threshold_calibration.json ml_pipeline/evaluation/reports/
mv kaggle_downloads/05_eer_calibration/pair_verification_roc_curves.png ml_pipeline/evaluation/reports/

# Then update shared_contracts/thresholds.json with the calibrated values from threshold_calibration.json
# Specifically: set match_threshold_value and flip match_threshold_status to "calibrated"

git add mobile_app/assets/models/ ml_pipeline/evaluation/reports/ shared_contracts/thresholds.json
git commit -m "ml(phase3): ship calibrated INT8 MobileFaceNet + ShuffleNet; threshold from EER"
git push
```

Tell teammate: **"threshold calibrated and final models shipped. Update the mobile loader's threshold constant from `match_threshold_value` in `shared_contracts/thresholds.json`."**